<a id="encrypted-machine-learning-utilities"></a>
# Encrypted Machine Learning: Utilities

This tutorial covers `src/concrete_fhe_toolkit/ml/utils.py`. This module provides handy array operations like one-hot encoding, binarizing, clipping, and normalizing that work directly on encrypted data.

<a id="encrypted-array-utilities"></a>
## Encrypted Array Utilities

Let's test `one_hot_encode`, `binarize`, and `clip_array` all inside a single circuit.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.utils import one_hot_encode, binarize, clip_array

def test_utils(label: int, feat1: int, feat2: int):
    ohe = one_hot_encode(label, num_classes=3)
    bin_feats = binarize([feat1, feat2], threshold=5)
    clipped = clip_array([feat1, feat2], min_val=0, max_val=10)
    
    return ohe, bin_feats, clipped

compiler = fhe.Compiler(test_utils, {"label": "encrypted", "feat1": "encrypted", "feat2": "encrypted"})

# Input bounds: Label between 0 and 2. Features can be negative or large.
circuit = compiler.compile([(0, -5, 15)])

ohe_res, bin_res, clipped_res = circuit.encrypt_run_decrypt(2, -5, 15)

# 1. One-hot encode label 2 for 3 classes -> [0, 0, 1]
assert list(ohe_res) == [0, 0, 1]

# 2. Binarize [-5, 15] with threshold 5 -> [-5 >= 5 (0), 15 >= 5 (1)] -> [0, 1]
assert list(bin_res) == [0, 1]

# 3. Clip [-5, 15] between 0 and 10 -> [0, 10]
assert list(clipped_res) == [0, 10]

print("✅ Encrypted ML Utilities passed!")